# Regressão Logística — SMOTE-NC
Compara três proporções de balanceamento (1:1 / 1:5 / 1:10) treinando com 2020–2023 e testando em 2024.  
Sem `class_weight='balanced'` — o rebalanceamento já está no dado.  
Métricas prioritárias: **Sensibilidade > AUPRC > ROC-AUC > Especificidade > F1**

In [ ]:
import json
import os
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)

BASE_DIR     = '../../data/features'
OUTPUT_MOD   = '../../output/modelos'
OUTPUT_MET   = '../../output/metricas'
ALGO         = 'logistic_regression'
RANDOM_STATE = 42

CONTINUOUS_COLS = ['age_years', 'epi_week', 'CS_GESTANT', 'CS_RACA', 'CS_ESCOL_N']

# Proporção minoritária:majoritária → sampling_strategy usada no balance_data.py
DATASETS = {
    'smote_1_1':  'smote_nc_1_1',
    'smote_1_5':  'smote_nc_1_5',
    'smote_1_10': 'smote_nc_1_10',
}

## 1. Funções auxiliares

In [ ]:
def calcular_metricas(y_true, y_pred_proba, threshold=0.5, label=''):
    y_pred = (y_pred_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'dataset':        label,
        'sensibilidade':  round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0,
        'especificidade': round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0,
        'auprc':          round(average_precision_score(y_true, y_pred_proba), 4),
        'roc_auc':        round(roc_auc_score(y_true, y_pred_proba), 4),
        'f1':             round(f1_score(y_true, y_pred), 4),
        'precisao':       round(precision_score(y_true, y_pred, zero_division=0), 4),
        'threshold':      threshold,
    }


def build_pipeline(feature_cols):
    """Pipeline para dados SMOTE já pré-processados: apenas escala colunas contínuas."""
    cont_present = [c for c in CONTINUOUS_COLS if c in feature_cols]
    preprocessor = ColumnTransformer(
        [('scale', StandardScaler(), cont_present)],
        remainder='passthrough',
        verbose_feature_names_out=False,
    )
    return Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
            solver='lbfgs',
        )),
    ])


print('Funções definidas.')

## 2. Treinamento e avaliação por proporção

In [ ]:
resultados = {}
probas     = {}   # label → (y_te, proba, pipeline)

# Test set é idêntico nos três diretórios — carregado uma vez
first_dir = os.path.join(BASE_DIR, list(DATASETS.values())[0])
y_test_raw = pd.read_parquet(os.path.join(first_dir, 'y_test.parquet')).squeeze()
X_test_raw = pd.read_parquet(os.path.join(first_dir, 'X_test.parquet'))

y_te_mask = y_test_raw.notna()
y_te      = y_test_raw[y_te_mask]
X_te      = X_test_raw[y_te_mask.values]

print(f'Teste (2024): {len(X_te):,} registros | Óbitos: {int(y_te.sum()):,} ({y_te.mean()*100:.2f}%)\n')

for label, dataset_dir in DATASETS.items():
    path    = os.path.join(BASE_DIR, dataset_dir)
    X_train = pd.read_parquet(os.path.join(path, 'X_train.parquet'))
    y_train = pd.read_parquet(os.path.join(path, 'y_train.parquet')).squeeze()

    with open(os.path.join(path, 'config.json')) as f:
        cfg = json.load(f)

    pipe = build_pipeline(X_train.columns.tolist())
    pipe.fit(X_train, y_train)

    proba = pipe.predict_proba(X_te)[:, 1]

    metricas = calcular_metricas(y_te, proba, label=label)
    metricas['n_train']         = len(X_train)
    metricas['n_obito_train']   = int(y_train.sum())
    metricas['ratio']           = cfg.get('ratio', label)
    metricas['sampling_strategy'] = cfg.get('sampling_strategy')

    resultados[label] = metricas
    probas[label]     = (y_te, proba, pipe)

    print(f'[{label}]  Treino: {len(X_train):,} | Óbitos treino: {int(y_train.sum()):,}')
    print(f'          Sensibilidade={metricas["sensibilidade"]:.4f} | '
          f'AUPRC={metricas["auprc"]:.4f} | ROC-AUC={metricas["roc_auc"]:.4f}\n')

## 3. Comparação de métricas — SMOTE vs. Baseline

In [ ]:
# Carrega resultado do baseline (modelo treinado com 2020-2023, sem SMOTE)
baseline_path = os.path.join(OUTPUT_MET, 'logistic_regression_baseline.parquet')
df_baseline   = pd.read_parquet(baseline_path)
row_baseline  = df_baseline[df_baseline['label'] == '2020+2021+2022+2023'].iloc[0]

baseline_entry = {
    'dataset':          'baseline',
    'ratio':            'sem SMOTE',
    'sensibilidade':    row_baseline['sensibilidade'],
    'especificidade':   row_baseline['especificidade'],
    'auprc':            row_baseline['auprc'],
    'roc_auc':          row_baseline['roc_auc'],
    'f1':               row_baseline['f1'],
    'precisao':         row_baseline['precisao'],
    'n_train':          row_baseline['n_train'],
    'n_obito_train':    row_baseline['n_obito_train'],
    'threshold':        row_baseline['threshold'],
}

df_comp = pd.DataFrame([baseline_entry] + list(resultados.values()))
cols_display = ['dataset', 'ratio', 'n_train', 'n_obito_train',
                'sensibilidade', 'especificidade', 'auprc', 'roc_auc', 'f1', 'precisao']
display(df_comp[cols_display])

## 4. Curvas ROC e Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cores = {'smote_1_1': '#C44E52', 'smote_1_5': '#DD8452', 'smote_1_10': '#4C72B0'}

for label, (y_true, proba, _) in probas.items():
    RocCurveDisplay.from_predictions(
        y_true, proba, ax=axes[0], name=label, color=cores[label]
    )
    PrecisionRecallDisplay.from_predictions(
        y_true, proba, ax=axes[1], name=label, color=cores[label]
    )

axes[0].set_title('Curva ROC — teste 2024')
axes[1].set_title('Curva Precision-Recall — teste 2024')
plt.tight_layout()
plt.show()

## 5. Matrizes de Confusão (threshold = 0.5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for col, (label, (y_true, proba, _)) in enumerate(probas.items()):
    y_pred = (proba >= 0.5).astype(int)
    cm     = confusion_matrix(y_true, y_pred)

    # Contagens
    ConfusionMatrixDisplay(cm, display_labels=['Cura', 'Óbito']).plot(
        ax=axes[0, col], colorbar=False, cmap='Blues'
    )
    axes[0, col].set_title(f'{label} — contagens')

    # Normalizada por linha
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    ConfusionMatrixDisplay(cm_norm, display_labels=['Cura', 'Óbito']).plot(
        ax=axes[1, col], colorbar=False, cmap='Blues'
    )
    axes[1, col].set_title(f'{label} — normalizada')
    for text in axes[1, col].texts:
        text.set_text(f'{float(text.get_text()):.2%}')

plt.suptitle('Matrizes de Confusão por proporção SMOTE (threshold = 0.5)', y=1.01)
plt.tight_layout()
plt.show()

## 6. Análise de threshold — proporção com maior sensibilidade

In [ ]:
# Identifica a proporção com maior sensibilidade em threshold=0.5
melhor_label = max(resultados, key=lambda k: resultados[k]['sensibilidade'])
print(f'Proporção com maior sensibilidade: {melhor_label}')

y_te_best, proba_best, _ = probas[melhor_label]

thresholds = np.arange(0.05, 0.96, 0.05)
rows = []

for t in thresholds:
    y_pred = (proba_best >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te_best, y_pred).ravel()
    rows.append({
        'threshold':      round(t, 2),
        'sensibilidade':  round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0,
        'especificidade': round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0,
        'precisao':       round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0,
        'f1':             round(f1_score(y_te_best, y_pred), 4),
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
    })

df_thresh = pd.DataFrame(rows)
df_thresh['youden'] = df_thresh['sensibilidade'] + df_thresh['especificidade'] - 1
idx_youden = df_thresh['youden'].idxmax()

high_sens = df_thresh[df_thresh['sensibilidade'] >= 0.90]
idx_90    = high_sens['especificidade'].idxmax() if not high_sens.empty else None

print(f"Threshold padrão  (0.50): sens={df_thresh.loc[df_thresh['threshold']==0.50, 'sensibilidade'].values[0]:.4f} "
      f"| esp={df_thresh.loc[df_thresh['threshold']==0.50, 'especificidade'].values[0]:.4f}")
print(f"Threshold Youden  ({df_thresh.loc[idx_youden,'threshold']:.2f}): "
      f"sens={df_thresh.loc[idx_youden,'sensibilidade']:.4f} | esp={df_thresh.loc[idx_youden,'especificidade']:.4f}")
if idx_90 is not None:
    print(f"Threshold sens≥90% ({df_thresh.loc[idx_90,'threshold']:.2f}): "
          f"sens={df_thresh.loc[idx_90,'sensibilidade']:.4f} | esp={df_thresh.loc[idx_90,'especificidade']:.4f}")

display(df_thresh.drop(columns='youden'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(df_thresh['threshold'], df_thresh['sensibilidade'], 'o-', color='#C44E52', label='Sensibilidade')
axes[0].plot(df_thresh['threshold'], df_thresh['especificidade'], 'o-', color='#4C72B0', label='Especificidade')
axes[0].plot(df_thresh['threshold'], df_thresh['precisao'],       'o-', color='#55A868', label='Precisão')
axes[0].plot(df_thresh['threshold'], df_thresh['f1'],             'o-', color='#8172B2', label='F1')
axes[0].axvline(df_thresh.loc[idx_youden, 'threshold'], color='gray', linestyle='--',
                linewidth=0.9, label=f'Youden ({df_thresh.loc[idx_youden,"threshold"]:.2f})')
axes[0].axvline(0.5, color='black', linestyle=':', linewidth=0.9, label='Padrão (0.50)')
axes[0].set_title(f'Métricas por threshold — {melhor_label}')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(1 - df_thresh['especificidade'], df_thresh['sensibilidade'], 'o-', color='#C44E52')
for _, row in df_thresh.iterrows():
    axes[1].annotate(f"{row['threshold']:.2f}",
                     (1 - row['especificidade'], row['sensibilidade']),
                     textcoords='offset points', xytext=(4, 2), fontsize=6, color='gray')
axes[1].set_title('Sensibilidade × (1 − Especificidade)')
axes[1].set_xlabel('1 − Especificidade (FPR)')
axes[1].set_ylabel('Sensibilidade (TPR)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Salvamento

In [ ]:
os.makedirs(OUTPUT_MOD, exist_ok=True)
os.makedirs(OUTPUT_MET, exist_ok=True)

for label, (y_true, proba, pipe) in probas.items():
    dataset_suffix = label  # ex: smote_1_5

    model_path = os.path.join(OUTPUT_MOD, f'{ALGO}_{dataset_suffix}.joblib')
    joblib.dump(pipe, model_path)
    print(f'Modelo salvo: {model_path}')

    met = resultados[label].copy()
    df_met = pd.DataFrame([met])
    met_path = os.path.join(OUTPUT_MET, f'{ALGO}_{dataset_suffix}.parquet')
    df_met.to_parquet(met_path, index=False)
    print(f'Métricas salvas: {met_path}')

    df_pred = pd.DataFrame({'y_true': y_true.values, 'y_proba': proba})
    pred_path = os.path.join(OUTPUT_MET, f'{ALGO}_{dataset_suffix}_predicoes.parquet')
    df_pred.to_parquet(pred_path, index=False)
    print(f'Predições salvas: {pred_path}\n')

## 8. Conclusão

O SMOTE-NC **não melhorou** o desempenho da regressão logística em nenhuma das proporções testadas.

| Dataset    | Sensibilidade | AUPRC  | ROC-AUC |
|------------|--------------|--------|---------|
| Baseline   | **0.7987**   | **0.6250** | **0.9240** |
| SMOTE 1:1  | 0.7645       | 0.5363 | 0.8367  |
| SMOTE 1:5  | 0.6042       | 0.5536 | 0.8553  |
| SMOTE 1:10 | 0.5428       | 0.5706 | 0.8718  |

**Observações:**
- A sensibilidade — métrica prioritária do projeto — cai progressivamente com o aumento do balanceamento.
- AUPRC e ROC-AUC também pioram em todas as variantes SMOTE, indicando calibração de probabilidade mais fraca.
- O SMOTE troca sensibilidade por precisão e especificidade, o que é o **inverso** do objetivo clínico (minimizar falsos negativos).

**Por que o baseline supera o SMOTE aqui:**
A regressão logística com `class_weight='balanced'` já compensa o desbalanceamento via pesos na função de perda, sem distorcer a distribuição dos dados. O SMOTE gera amostras sintéticas por interpolação, o que pode introduzir ruído em features binárias e categóricas mesmo com SMOTE-NC.

**Modelo selecionado para regressão logística:** `logistic_regression_baseline` (sem SMOTE, `class_weight='balanced'`).